# J15-B — Qualification reproductible de la réallocation OR-Tools

Ce notebook reproduit le benchmark synthétique préenregistré de réallocation consultative avec `SimpleMinCostFlow`. Il est épinglé au commit qualifié `f71a80ac657c5ed58a8147e8535bdba60dddde0d`.

Il ne mesure pas une performance locale RentFleet, n'utilise aucune sortie CatBoost et n'autorise aucune écriture métier automatique.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

REPOSITORY_URL = "https://github.com/getibplay-cmyk/pfe.git"
QUALIFICATION_COMMIT = "f71a80ac657c5ed58a8147e8535bdba60dddde0d"
BENCHMARK_SHA256 = "53d79202807b2952dc95154e0116153664f202007807a0855a16cbea63cc4214"
PASS_DECISION = "QUALIFIED_FOR_CONSULTATIVE_SAAS_INTEGRATION_REVIEW"
WORKSPACE = Path(tempfile.mkdtemp(prefix="rentfleet-j15b-"))
REPOSITORY = WORKSPACE / "pfe"
OUTPUT = WORKSPACE / "qualification-output"


## 1. Charger exactement le commit qualifié

Le checkout détaché interdit qu'une évolution ultérieure de la branche modifie silencieusement la reproduction.

In [ ]:
subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--quiet", "--detach", QUALIFICATION_COMMIT], check=True)
checked_out = subprocess.check_output(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True
).strip()
assert checked_out == QUALIFICATION_COMMIT
print(f"Commit qualifié chargé : {checked_out}")


## 2. Installer l'environnement OR-Tools figé

Les dépendances directes et transitives proviennent uniquement du fichier versionné du commit qualifié.

In [ ]:
requirements = REPOSITORY / "scripts/intelligence/requirements-fleet-reallocation.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--requirement", str(requirements)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)


## 3. Réexécuter la qualification préenregistrée

Le seed et `PYTHONHASHSEED` sont gelés à `20260814`. Les temps solveur restent dépendants de la machine et ne sont pas interprétés comme une promesse de production.

In [ ]:
environment = os.environ.copy()
environment["PYTHONHASHSEED"] = "20260814"
subprocess.run(
    [
        sys.executable,
        str(REPOSITORY / "scripts/intelligence/qualify_fleet_reallocation.py"),
        "--output",
        str(OUTPUT),
    ],
    check=True,
    cwd=REPOSITORY,
    env=environment,
)


## 4. Vérifier les empreintes et les résultats

Chaque fichier produit est contrôlé par SHA-256. Les artefacts déterministes doivent être identiques, octet pour octet, aux preuves qualifiées du dépôt.

In [ ]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

checksums = {}
for line in (OUTPUT / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
    digest, name = line.split("  ", maxsplit=1)
    checksums[name] = digest

produced = {path.name for path in OUTPUT.iterdir() if path.is_file() and path.name != "SHA256SUMS"}
assert produced == set(checksums)
for name, digest in checksums.items():
    assert sha256_file(OUTPUT / name) == digest, name

frozen_evidence = REPOSITORY / "docs/evidence/intelligence/fleet-reallocation"
deterministic_artifacts = {
    "benchmark-comparison.svg",
    "benchmark-scenarios.json",
    "method-summary.csv",
    "qualification-manifest.json",
    "sample-human-review-plan.csv",
    "scenario-results.csv",
}
for name in deterministic_artifacts:
    assert (OUTPUT / name).read_bytes() == (frozen_evidence / name).read_bytes(), name


In [ ]:
manifest = json.loads((OUTPUT / "qualification-manifest.json").read_text(encoding="utf-8"))
assert manifest["decision"] == PASS_DECISION
assert manifest["gate_passed"] is True
assert manifest["benchmark"]["snapshot_sha256"] == BENCHMARK_SHA256
assert manifest["benchmark"]["scenario_count"] == 48
assert manifest["aggregates"]["ortools_min_cost_flow"]["service_rate"] == "0.983607"
assert manifest["aggregates"]["ortools_min_cost_flow"]["unserved_demand"] == 12
assert manifest["aggregates"]["greedy"]["unserved_demand"] == 180
assert manifest["aggregates"]["no_relocation"]["unserved_demand"] == 348
assert manifest["safety"]["catboost_output_consumed"] is False
assert manifest["safety"]["automatic_business_write_allowed"] is False
print("Qualification reproduite :", manifest["decision"])


## Lecture correcte du résultat

Le gate scientifique est réussi sur 48 scénarios synthétiques : 720 demandes servies sur 732 et 12 pénuries inévitables. Cette preuve autorise uniquement une revue d'intégration SaaS consultative. Elle ne démontre ni précision locale, ni économie réelle en MAD, ni droit d'exécuter automatiquement une réallocation.